# Data Preprocessing

Following the exploratory data analysis, the dataset is now prepared for machine learning.

The preprocessing stage will focus on:

1. Creating a clean working copy of the dataset.
2. Removing duplicate records.
3. Converting and validating the date variable.
4. Handling missing values.
5. Checking numerical feature ranges and invalid values.
6. Preparing the target variable.
7. Separating predictors and target.
8. Preparing a leakage-safe preprocessing pipeline for model development.

The original dataset will not be modified. All preprocessing operations will be performed on a working copy.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# ============================================================
# LOAD MESOGEOS DATASET
# ============================================================

DATASET_PATH = Path("../../dataset/mesogeos_wildfire_dataset.csv")

df = pd.read_csv(DATASET_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

Dataset loaded successfully.
Dataset shape: (11305, 32)


In [3]:
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())


Columns:
['date', 'latitude', 'longitude', 'temperature_c', 'dew_point_c', 'relative_humidity', 'wind_speed', 'wind_direction', 'rainfall_mm', 'surface_pressure', 'solar_radiation', 'ndvi', 'lai', 'soil_moisture', 'elevation', 'slope_degrees', 'aspect', 'curvature', 'roads_distance_km', 'population', 'lc_agriculture', 'lc_forest', 'lc_grassland', 'lc_settlement', 'lc_shrubland', 'lc_sparse_vegetation', 'lc_water_bodies', 'lc_wetland', 'burned_area_ha', 'year', 'month', 'day_of_year']

First 5 records:


,date,latitude,longitude,temperature_c,dew_point_c,relative_humidity,wind_speed,wind_direction,rainfall_mm,surface_pressure,...,lc_grassland,lc_settlement,lc_shrubland,lc_sparse_vegetation,lc_water_bodies,lc_wetland,burned_area_ha,year,month,day_of_year
0,2006-08-27,38.123648,-7.819233,36.419031,15.989252,22.510025,3.511700,329.388733,0.000000,99694.617188,...,0.0,0.0,0.000000,0.0,0.0,0.0,40.0,2006,8,239
1,2006-08-11,42.000202,0.447392,27.559045,7.369226,16.027991,6.567619,296.081879,0.004796,93547.523438,...,0.0,0.0,0.192824,0.0,0.0,0.0,1390.0,2006,8,223
2,2006-07-11,41.617581,-8.524061,34.471796,20.400690,35.828364,2.905688,345.955994,0.550396,99709.054688,...,0.0,0.0,0.000000,0.0,0.0,0.0,173.0,2006,7,192
3,2006-09-07,41.557167,-7.567509,30.666193,12.939386,28.009099,2.133366,313.057190,2.396404,93731.859375,...,0.0,0.0,0.000000,0.0,0.0,0.0,220.0,2006,9,250
4,2006-09-08,41.547098,-6.248474,31.690332,14.382166,22.408468,3.914824,288.407166,0.974770,93459.937500,...,0.0,0.0,0.000000,0.0,0.0,0.0,76.0,2006,9,251


In [4]:
df_clean = df.copy()

print("Original dataset shape:", df.shape)
print("Working dataset shape:", df_clean.shape)

Original dataset shape: (11305, 32)
Working dataset shape: (11305, 32)


## 1. Remove Duplicate Records

The exploratory data analysis identified two completely duplicated records.

Duplicate observations can give repeated events disproportionate influence during model training. Therefore, the duplicated records will be removed from the working dataset.

The original CSV file will remain unchanged.

In [5]:
duplicates_before = df_clean.duplicated().sum()

print("Duplicate records before removal:", duplicates_before)

df_clean = df_clean.drop_duplicates().reset_index(drop=True)

duplicates_after = df_clean.duplicated().sum()

print("Duplicate records after removal:", duplicates_after)
print("Clean dataset shape:", df_clean.shape)

Duplicate records before removal: 2
Duplicate records after removal: 0
Clean dataset shape: (11303, 32)


In [6]:
assert df_clean.duplicated().sum() == 0

print("Duplicate verification passed.")
print("Final unique records:", len(df_clean))

Duplicate verification passed.
Final unique records: 11303


## 2. Validate and Convert the Date Variable

The `date` variable is currently stored as a string. It will be converted to Pandas datetime format so that temporal features can be reliably extracted.

The conversion will also be used to identify invalid or incorrectly formatted dates.

The existing `year`, `month`, and `day_of_year` variables will subsequently be checked against the converted date to ensure temporal consistency.

In [7]:
print("Date data type:", df_clean["date"].dtype)

print("\nSample date values:")
display(df_clean["date"].head(10))

print("\nNumber of unique date values:", df_clean["date"].nunique())

Date data type: str

Sample date values:


0    2006-08-27
1    2006-08-11
2    2006-07-11
3    2006-09-07
4    2006-09-08
5    2006-07-11
6    2006-08-30
7    2006-09-04
8    2006-09-03
9    2006-06-21
Name: date, dtype: str


Number of unique date values: 2819


In [8]:
# CONVERT DATE TO DATETIME

df_clean["date"] = pd.to_datetime(
    df_clean["date"],
    errors="coerce"
)

print("Date data type after conversion:", df_clean["date"].dtype)

print(
    "\nInvalid dates created by conversion:",
    df_clean["date"].isna().sum()
)

Date data type after conversion: datetime64[us]

Invalid dates created by conversion: 0


In [9]:
# DATE RANGE CHECK

print("Earliest date:", df_clean["date"].min())
print("Latest date:", df_clean["date"].max())

print("\nNumber of records by year:")
display(
    df_clean["date"]
    .dt.year
    .value_counts()
    .sort_index()
)

Earliest date: 2006-04-09 00:00:00
Latest date: 2022-09-28 00:00:00

Number of records by year:


date
2006     214
2007     603
2008     296
2009     456
2010     420
2011     781
2012    1046
2013     424
2014     320
2015     460
2016     532
2017    1321
2018     357
2019     942
2020     987
2021     944
2022    1200
Name: count, dtype: int64